In [17]:
import sys

sys.path.append('/kaggle/working/vision-architectures-pytorch')


In [18]:

!git clone 'https://github.com/mohammadrezakasra/vision-architectures-pytorch.git'


import torch
import torch.nn as nn
from torch.utils.data import Subset , DataLoader , Dataset
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
import os
from PIL import Image
from models.cnn.KasraNet_V1_IncepNeXt import KasraNet_V1_IncepNeXt
from utils.trainer import Trainer

fatal: destination path 'vision-architectures-pytorch' already exists and is not an empty directory.


In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [20]:
torch.backends.cudnn.benchmark = True


In [21]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('cassava-leaf-disease-classification')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/cassava-leaf-disease-classification


In [22]:
train_transformer = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomAffine(degrees=15, shear=5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.0),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),(0.229, 0.224, 0.225))
])

val_transformer = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),(0.229, 0.224, 0.225))
])

In [23]:
train_csv = pd.read_csv("/kaggle/input/competitions/cassava-leaf-disease-classification/train.csv")
sample_csv= pd.read_csv('/kaggle/input/competitions/cassava-leaf-disease-classification/sample_submission.csv')

In [24]:
class CassavaDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.loc[idx, "image_id"]
        img_path = os.path.join(self.img_dir, img_id)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, img_id

        label = torch.tensor(self.df.loc[idx, "label"], dtype=torch.long)
        return image, label    

In [26]:
indices = list(range(train_csv.__len__()))
train_ind , val_ind = train_test_split(train_csv,test_size= 0.1,random_state=42 ,stratify=train_csv["label"])

In [27]:
train_df = train_ind.reset_index(drop=True)
val_df = val_ind.reset_index(drop=True)

In [28]:
trainset = CassavaDataset(train_df, os.path.join(path, "train_images"), transform=train_transformer, is_test=False)
valset   = CassavaDataset(val_df,   os.path.join(path, "train_images"), transform=val_transformer,   is_test=False)
testset  = CassavaDataset(sample_csv, os.path.join(path, "test_images"), transform=val_transformer,  is_test=True)

In [29]:
train_loader = DataLoader(
    trainset,
    batch_size= 64,
    shuffle= True,
    num_workers=2,
    pin_memory= True

)

val_loader = DataLoader(
    valset,
    batch_size= 64,
    shuffle= False,
    num_workers=2,
    pin_memory= True

)

testset_loader = DataLoader(
    testset,
    batch_size= 64,
    shuffle= False,
    num_workers=2,
    pin_memory= True

)

In [30]:
kasranet_v1_incepNeXt = KasraNet_V1_IncepNeXt(3,5,28,16,2).to(device)

In [31]:
optimizer_kasrav1 = torch.optim.SGD(kasranet_v1_incepNeXt.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


In [32]:
EPOCHS = 150
scheduler_kasrav1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_kasrav1, T_max=EPOCHS, eta_min=1e-6)

train_kv1 = Trainer(kasranet_v1_incepNeXt,optimizer_kasrav1,criterion,device,scheduler_kasrav1)


In [ ]:
history_kv1 = train_kv1.fit(train_loader, val_loader, EPOCHS, "kv1_Cassava_Leaf_Disease")

Starting training on device: cuda (GPUs: 2, AMP: True) for 150 epochs...



In [ ]:
print("len(trainset):", len(trainset))
print("len(valset):  ", len(valset))
print("train_df size:", len(train_df), " val_df size:", len(val_df))
print("trainset is valset:", trainset is valset)
print("overlap ids:  ", len(set(train_df.image_id) & set(val_df.image_id)))